# Historic Software Engineering: Insights from Deluxe Paint for the Amiga

Giuseppe Destefanis (University College London), Yann-Gaël Guéhéneuc (Concordia University), and Fabio Calefato (University of Bari)

---

**Replication notebook** for the architectural (RQ2), coding-idiom (RQ4), and complexity (RQ5) analyses in the paper. It performs software archaeology on the *Deluxe Paint V1* source code (Electronic Arts, 1985) for the Commodore Amiga (Motorola 68000 at 7.16 MHz, 256 KB RAM).

**Dataset:** 88 source files (77 `.C` + 11 `.H`) · 546 functions · 17,001 lines of code · total cyclomatic complexity 1,223.

The design-pattern analysis (RQ3) is verified separately by `verify_patterns.py`; the development and quality analysis (RQ1) is qualitative and reported in the paper.


## Research Questions

The paper answers five research questions:

- **RQ1.** How and by whom was *Deluxe Paint* developed, and what was its overall quality?
- **RQ2.** What architectural styles can be observed in the source code?
- **RQ3.** What design patterns are present in the source code?
- **RQ4.** What coding idioms are present in the source code?
- **RQ5.** How was complexity distributed across the different components, and why?

This notebook reproduces the quantitative results for **RQ2**, **RQ4**, and **RQ5**, in that order. Each section below is headed with the paper's question number:

- **RQ2 — architectural styles:** dependency graph, centrality, community detection, robustness.
- **RQ4 — coding idioms:** code, comment, and preprocessor density; coupling and instability.
- **RQ5 — complexity distribution:** file- and function-level cyclomatic complexity, functional categories, correlations.

**RQ1** (development and quality) is qualitative and reported in the paper. **RQ3** (design patterns) is verified by `verify_patterns.py`.

The metrics were extracted with Understand SciTools and are provided as CSV exports in `data/metrics/`.


## 1. Setup and Data Loading

We'll analyze the Deluxe Paint codebase using metrics extracted with Understand SciTools.

In [ ]:
# Install dependencies (Colab or local). Versions pinned for reproducibility;
# plotly is pinned to 5.x because the figures use the 5.x colorbar API.
!pip install -q pandas numpy networkx scipy matplotlib seaborn "plotly==5.24.1" scikit-learn statsmodels


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import spearmanr, pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("✓ Libraries loaded successfully")
print("\nReady to analyze Deluxe Paint (1985) source code")
print("Platform: Amiga 1000 | RAM: 256KB | Language: C")

### Loading the data

The next cell loads the metric exports automatically:

- **`file.csv`** — file-level metrics (88 files)
- **`function.csv`** — function-level metrics (546 functions)
- **`source_FileDependencies.csv`** — file-to-file dependency edges

Run locally from the repository root (the cell locates `data/metrics/` automatically). On Google Colab it falls back to uploading these three files.


In [ ]:
# ---------------------------------------------------------------------------
# Load the metric exports. Runs locally (replication package) or on Colab.
#   Local : reads the shipped CSVs from data/metrics/.
#   Colab : falls back to files.upload() (upload file.csv, function.csv,
#           source_FileDependencies.csv).
# The raw Understand exports do not carry the derived columns the analysis uses
# (ComplexityRisk, DependencyCount, Module); we derive them here.
# ---------------------------------------------------------------------------
import os
import pandas as pd

def _find_metrics_dir():
    for d in ['data/metrics', '../data/metrics', 'metrics', '.']:
        if os.path.exists(os.path.join(d, 'file.csv')):
            return d
    return None

_mdir = _find_metrics_dir()
if _mdir is None:
    from google.colab import files
    print("Upload file.csv, function.csv, source_FileDependencies.csv ...")
    files.upload()
    _mdir = '.'

file_complexity     = pd.read_csv(os.path.join(_mdir, 'file.csv'))
function_complexity = pd.read_csv(os.path.join(_mdir, 'function.csv'))
file_dependencies   = pd.read_csv(os.path.join(_mdir, 'source_FileDependencies.csv'))

# --- Derived columns (absent from the raw Understand exports) ---
def _risk(cc):                      # McCabe complexity-risk bands (function level)
    if cc <= 10: return 'Low'
    if cc <= 20: return 'Medium'
    if cc <= 50: return 'High'
    return 'Very High'
function_complexity['ComplexityRisk'] = function_complexity['Cyclomatic'].apply(_risk)
file_complexity['Module'] = file_complexity['Name'].str.extract(r'^([A-Z]+)')[0]

# Internal dependency graph: keep only edges whose target is a known DPaint file
# (drops external Amiga-OS headers such as exec/types.h). Graph
# metrics are computed on this internal graph (70 nodes / 366 edges).
_known = set(file_complexity['Name'])
internal_dependencies = file_dependencies[file_dependencies['Target'].isin(_known)].copy()
dependencies = (internal_dependencies.groupby('Source').size()
                .reset_index(name='DependencyCount'))

print("="*60); print("DATA LOADED"); print("="*60)
print(f"  Files analysed:              {len(file_complexity)}")
print(f"  Functions identified:        {len(function_complexity)}")
print(f"  Dependency edges (total):    {len(file_dependencies)}")
print(f"  Dependency edges (internal): {len(internal_dependencies)}")
print(f"  Total lines of code:         {file_complexity['CountLine'].sum():,}")
print(f"  Total cyclomatic complexity: {file_complexity['SumCyclomatic'].sum():,}")

# Functional category of each source file (by primary purpose), loaded as data.
_cat = pd.read_csv(os.path.join(_mdir, 'file_categories.csv'))
file_complexity = file_complexity.merge(_cat, on='Name', how='left')


## 2. Dataset Overview & Historical Context

Before answering the research questions, we summarise the characteristics of this 1985 codebase.

In [ ]:
# Display dataset structure
print("FILE-LEVEL METRICS (Sample)")
print("="*80)
print(file_complexity[[
    'Name', 'CountLine', 'CountDeclFunction', 'SumCyclomatic',
    'RatioCommentToCode'
]].head(10))

print("\n" + "="*80)
print("FUNCTION-LEVEL METRICS (Sample)")
print("="*80)
print(function_complexity[[
    'Name', 'CountLine', 'Cyclomatic', 'MaxNesting', 'ComplexityRisk'
]].head(10))

print("\n" + "="*80)
print("KEY STATISTICS")
print("="*80)
print(f"Total LOC: {file_complexity['CountLine'].sum():,}")
print(f"Total executable LOC: {file_complexity['CountLineCode'].sum():,}")
print(f"Total cyclomatic complexity: {file_complexity['SumCyclomatic'].sum():,}")
print(f"Average comment density: {(file_complexity['CountLineComment'] / file_complexity['CountLine']).mean():.2f}")
print(f"\nFunctions by complexity risk:")
print(function_complexity['ComplexityRisk'].value_counts().sort_index())

---
## RQ2: Architectural styles

**What architectural styles can be observed in the source code of *Deluxe Paint*?**

*Deluxe Paint* was written in 1985 without design-pattern literature, object-oriented standards, UML, or online developer communities. We examine the architectural structure that emerged from the problem domain and the hardware constraints, using the file dependency graph: centrality, community detection, and robustness.

---


In [ ]:
# =============================================================================
# RQ2: Build Dependency Network Graph
# This cell constructs the graph used by all subsequent RQ2 visualisations
# =============================================================================

import networkx as nx

print("Building dependency network from file relationships...")
print()

# Build directed graph from the INTERNAL file dependencies (external Amiga-OS
# headers excluded), so the centrality, community, robustness, and instability
# results below reflect DPaint's own inter-file dependencies (70 nodes / 366 edges).
G = nx.DiGraph()
for idx, row in internal_dependencies.iterrows():
    G.add_edge(row['Source'], row['Target'], weight=row['weight'])

# Calculate centrality metrics (used by multiple visualisations)
betweenness = nx.betweenness_centrality(G)
degree_dict = dict(G.degree())
in_degree_dict = dict(G.in_degree())
out_degree_dict = dict(G.out_degree())

# Create undirected version for some analyses
G_undirected = G.to_undirected()

# Calculate layout (shared across 2D visualisations)
pos = nx.spring_layout(G, k=1.5, iterations=100, seed=42, weight='weight')
pos_3d = nx.spring_layout(G, dim=3, k=1.5, iterations=100, seed=42, weight='weight')

print(f"{'='*60}")
print("DEPENDENCY GRAPH CONSTRUCTED")
print(f"{'='*60}")
print(f"  Nodes (files): {G.number_of_nodes()}")
print(f"  Edges (dependencies): {G.number_of_edges()}")
print(f"  Network density: {nx.density(G):.4f}")

# Identify hub files (high in-degree = many files depend on them)
hub_threshold = 10
hub_files = [(n, in_degree_dict[n]) for n in G.nodes() if in_degree_dict[n] >= hub_threshold]
hub_files = sorted(hub_files, key=lambda x: x[1], reverse=True)

print(f"\n  Hub files (≥{hub_threshold} incoming dependencies): {len(hub_files)}")
for name, deg in hub_files[:5]:
    print(f"    {name:25s} ← {deg:2d} files")

# Identify bridge files (high betweenness = connect different parts)
bridge_files = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]

print(f"\n  Top architectural bridges (betweenness centrality):")
for name, bc in bridge_files:
    print(f"    {name:25s}   {bc:.4f}")

print(f"\n✓ Graph ready for analysis")

In [ ]:
# RQ2: Dependency Network Analysis
print("RQ2: ARCHITECTURAL STYLE ANALYSIS")
print("="*80)
print("Analysing the dependency structure to identify the architectural style...")
print()

# Basic dependency statistics
avg_deps = dependencies['DependencyCount'].mean()
median_deps = dependencies['DependencyCount'].median()
max_deps = dependencies['DependencyCount'].max()
highly_connected = len(dependencies[dependencies['DependencyCount'] > 15])
loosely_connected = len(dependencies[dependencies['DependencyCount'] < 5])

print(f"Dependency Statistics:")
print(f"  Average dependencies per file: {avg_deps:.1f}")
print(f"  Median dependencies: {median_deps:.0f}")
print(f"  Maximum dependencies: {max_deps}")
print(f"  Highly connected files (>15 deps): {highly_connected}")
print(f"  Loosely connected files (<5 deps): {loosely_connected}")

# Identify architectural style
print(f"\nArchitectural Style Detection:")
if highly_connected > 0 and loosely_connected > highly_connected * 3:
    print("  ✓ HUB-AND-SPOKE ARCHITECTURE detected")
    print("    - Few central 'hub' files with many dependencies")
    print("    - Many peripheral 'spoke' files with few dependencies")
    print("    - This emerged naturally without formal design patterns!")
else:
    print("  Style: Distributed/Modular Architecture")

# Identify hub files
print(f"\nHub Files (Central Coordination):")
hubs = dependencies.nlargest(5, 'DependencyCount')[['Source', 'DependencyCount']]
for idx, row in hubs.iterrows():
    print(f"  {row['Source']:20s} → {row['DependencyCount']:2d} dependencies")

In [ ]:
# RQ2: Visualize dependency network
print("\nVisualizing dependency network structure...")

# Merge with complexity data
dep_complexity = dependencies.merge(
    file_complexity[['Name', 'SumCyclomatic', 'CountLine']],
    left_on='Source',
    right_on='Name',
    how='left'
)

# Scatter plot: Dependencies vs Complexity
fig = px.scatter(
    dep_complexity.head(40),
    x='DependencyCount',
    y='SumCyclomatic',
    size='CountLine',
    hover_data=['Source'],
    title='RQ2: Dependency Count vs Cyclomatic Complexity<br><sub>Do hub files exhibit higher complexity?</sub>',
    labels={
        'DependencyCount': 'Number of Dependencies (Coupling)',
        'SumCyclomatic': 'Total Cyclomatic Complexity',
        'CountLine': 'File Size (LOC)'
    },
    color='DependencyCount',
    color_continuous_scale='Reds',
    trendline='ols'
)

fig.update_layout(height=600)
fig.show()

# Statistical test
corr, pval = spearmanr(dep_complexity['DependencyCount'].dropna(),
                       dep_complexity['SumCyclomatic'].dropna())
print(f"\nSpearman correlation (Dependencies vs Complexity): {corr:.3f}")
print(f"P-value: {pval:.4f} {'(significant)' if pval < 0.05 else '(not significant)'}")

if corr > 0.3:
    print("\n✓ Finding: Hub files show HIGHER complexity")
    print("  Interpretation: Central coordination files handle more logic")
else:
    print("\nFinding: Complexity is distributed, not centralized")

In [ ]:
# RQ2: Module Detection (organic grouping)
print("\nRQ2: MODULE DETECTION")
print("="*60)
print("Identifying natural module groupings from file naming conventions...\n")

# Extract module prefixes (e.g., PAL from PALETTE.C)
file_complexity['Module'] = file_complexity['Name'].str.extract(r'^([A-Z]+)')[0]

# Count modules
module_counts = file_complexity.groupby('Module').agg({
    'Name': 'count',
    'SumCyclomatic': 'sum',
    'CountLine': 'sum'
}).rename(columns={'Name': 'Files'}).sort_values('Files', ascending=False)

print("Top 10 Functional Modules (by file count):")
print(module_counts.head(10))

print(f"\n✓ Total distinct modules identified: {len(module_counts)}")
print("\nInterpretation: Even without formal design, developers organized")
print("code into logical functional modules (palette, display, I/O, etc.)")

### RQ2 Visualization 1: 2D Dependency Network

Interactive network graph showing file dependencies and architectural structure.

In [ ]:
# =============================================================================
# RQ2 Visualization 1: Interactive Dependency Network
# Uses: G, pos, betweenness, in_degree_dict, out_degree_dict (from Step 1)
# =============================================================================

import plotly.graph_objects as go

# Get edge weights for thickness scaling
weights = [G[u][v]['weight'] for u, v in G.edges()]
max_weight = max(weights) if weights else 1

# Create edge traces with varying thickness
edge_traces = []
for edge in G.edges(data=True):
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    weight = edge[2]['weight']

    # Scale thickness (0.5 to 4 based on weight)
    thickness = 0.5 + (weight / max_weight) * 3.5

    edge_trace = go.Scatter(
        x=[x0, x1, None],
        y=[y0, y1, None],
        line=dict(width=thickness, color='rgba(150,150,150,0.3)'),
        hoverinfo='none',
        mode='lines',
        showlegend=False
    )
    edge_traces.append(edge_trace)

# Prepare node data
node_x = []
node_y = []
node_text = []
node_size = []
node_color = []
node_labels = []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)

    # Get metrics from pre-computed dictionaries
    degree = degree_dict.get(node, 0)
    between = betweenness.get(node, 0)
    in_deg = in_degree_dict.get(node, 0)
    out_deg = out_degree_dict.get(node, 0)

    # Hover text
    hover = (f"<b>{node}</b><br>"
             f"Total connections: {degree}<br>"
             f"Incoming: {in_deg}<br>"
             f"Outgoing: {out_deg}<br>"
             f"Betweenness: {between:.4f}")
    node_text.append(hover)

    # Node size based on degree
    node_size.append(max(8, degree * 1.5))

    # Node colour based on betweenness centrality
    node_color.append(between)

    # Short label
    label = node.replace('.C', '').replace('.H', '').replace('exec\\', '').replace('librarie\\', '')
    node_labels.append(label[:10])

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    hoverinfo='text',
    hovertext=node_text,
    text=node_labels,
    textposition='top center',
    textfont=dict(size=7, color='#444444'),
    marker=dict(
        showscale=True,
        colorscale='YlOrRd',
        size=node_size,
        color=node_color,
        colorbar=dict(
            thickness=15,
            title='Betweenness<br>Centrality',
            xanchor='left',
            titleside='right',
            len=0.5
        ),
        line=dict(width=1, color='white')
    )
)

# Create the figure
fig = go.Figure(
    data=edge_traces + [node_trace],
    layout=go.Layout(
        title=dict(
            text='Dependency Network',
            x=0.5,
            xanchor='center',
            font=dict(size=18)
        ),
        showlegend=False,
        hovermode='closest',
        margin=dict(b=20, l=20, r=20, t=100),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=750,
        paper_bgcolor='white',
        plot_bgcolor='white',
        annotations=[
            dict(
                x=0.5, y=1.12,
                xref='paper', yref='paper',
                text='<b>How to read this network:</b><br>'
                     '• <b>Nodes:</b> Source files (size indicates how connected they are)<br>'
                     '• <b>Edges:</b> Dependencies between files (thickness shows strength)<br>'
                     '• <b>Colour:</b> Betweenness centrality (files that bridge different parts of the system)<br>'
                     '• <b>Key files:</b> Large, highly coloured nodes are critical to the architecture',
                showarrow=False,
                font=dict(size=11),
                align='center',
                bgcolor='rgba(240,245,250,0.9)',
                bordercolor='#cccccc',
                borderwidth=1,
                borderpad=8,
                xanchor='center',
                yanchor='bottom'
            )
        ]
    )
)

fig.show()

# Summary statistics
print(f"\nTop 10 most depended-upon files:")
in_degrees_sorted = sorted(in_degree_dict.items(), key=lambda x: x[1], reverse=True)
for name, deg in in_degrees_sorted[:10]:
    print(f"  {name:25s} ← {deg:2d} files depend on this")

### RQ2 Visualization 2: 3D Dependency Network

Three-dimensional network visualization for exploring architectural layers.

In [ ]:
# =============================================================================
# RQ2 Visualization 2: 3D Dependency Network
# Uses: G, pos_3d, betweenness, in_degree_dict (from Step 1)
# =============================================================================

import plotly.graph_objects as go

# Create 3D edge traces
edge_x_3d = []
edge_y_3d = []
edge_z_3d = []

for edge in G.edges():
    x0, y0, z0 = pos_3d[edge[0]]
    x1, y1, z1 = pos_3d[edge[1]]
    edge_x_3d.extend([x0, x1, None])
    edge_y_3d.extend([y0, y1, None])
    edge_z_3d.extend([z0, z1, None])

edge_trace_3d = go.Scatter3d(
    x=edge_x_3d, y=edge_y_3d, z=edge_z_3d,
    mode='lines',
    line=dict(color='rgba(150,150,150,0.4)', width=1.5),
    hoverinfo='none'
)

# Create 3D node traces
node_x_3d = []
node_y_3d = []
node_z_3d = []
node_text_3d = []
node_size_3d = []
node_color_3d = []

for node in G.nodes():
    x, y, z = pos_3d[node]
    node_x_3d.append(x)
    node_y_3d.append(y)
    node_z_3d.append(z)

    # Get metrics from pre-computed dictionaries
    degree = degree_dict.get(node, 0)
    between = betweenness.get(node, 0)
    in_deg = in_degree_dict.get(node, 0)
    out_deg = out_degree_dict.get(node, 0)

    # Hover text
    hover = (f"{node}<br>"
             f"Connections: {degree}<br>"
             f"In: {in_deg} | Out: {out_deg}<br>"
             f"Betweenness: {between:.4f}")
    node_text_3d.append(hover)

    # Node size based on total degree
    node_size_3d.append(max(4, degree * 1.2))

    # Node colour based on betweenness centrality
    node_color_3d.append(between)

node_trace_3d = go.Scatter3d(
    x=node_x_3d, y=node_y_3d, z=node_z_3d,
    mode='markers',
    marker=dict(
        size=node_size_3d,
        color=node_color_3d,
        colorscale='YlOrRd',
        showscale=True,
        colorbar=dict(
            title='Betweenness<br>Centrality',
            thickness=15,
            len=0.5
        ),
        line=dict(color='white', width=0.5)
    ),
    text=node_text_3d,
    hoverinfo='text'
)

fig_3d = go.Figure(data=[edge_trace_3d, node_trace_3d])
fig_3d.update_layout(
    title=dict(
        text='3D Dependency Network<br><sup>Rotate and zoom to explore architectural layers</sup>',
        x=0.5,
        xanchor='center'
    ),
    scene=dict(
        xaxis=dict(showbackground=False, showticklabels=False, title=''),
        yaxis=dict(showbackground=False, showticklabels=False, title=''),
        zaxis=dict(showbackground=False, showticklabels=False, title=''),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    height=700,
    showlegend=False,
    paper_bgcolor='white'
)

fig_3d.show()

print("✓ 3D network reveals architectural layers")
print(f"  Rotate to explore clustering of related files")
print(f"  Hub files (SYSTEM.H, PRISM.H) should appear central with many connections")

### RQ2 Visualization 3: Centrality Analysis

Graph theory metrics identifying the most critical files in the architecture.

In [ ]:
# =============================================================================
# RQ2 Visualization 3: Centrality Analysis
# Uses: G, betweenness, degree_dict, in_degree_dict (from Step 1)
# =============================================================================

print("Analysing graph centrality metrics...")
print()

# Degree centrality (normalised)
degree_cent = nx.degree_centrality(G)

# Eigenvector centrality (files connected to important files)
try:
    eigenvector = nx.eigenvector_centrality(G, max_iter=1000)
except:
    eigenvector = {node: 0 for node in G.nodes()}

# PageRank (alternative importance measure)
pagerank = nx.pagerank(G)

# Create DataFrame with all centrality measures
centrality_df = pd.DataFrame({
    'File': list(G.nodes()),
    'Betweenness': [betweenness[n] for n in G.nodes()],
    'Degree': [degree_cent[n] for n in G.nodes()],
    'In-Degree': [in_degree_dict[n] for n in G.nodes()],
    'Out-Degree': [out_degree_dict[n] for n in G.nodes()],
    'Eigenvector': [eigenvector.get(n, 0) for n in G.nodes()],
    'PageRank': [pagerank.get(n, 0) for n in G.nodes()]
})

# Sort by betweenness (most important for identifying architectural bridges)
centrality_df = centrality_df.sort_values('Betweenness', ascending=False)

# Create subplot with three key centrality measures
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        'Betweenness Centrality<br><sup>(Bridge files)</sup>',
        'In-Degree<br><sup>(Most depended upon)</sup>',
        'PageRank<br><sup>(Overall importance)</sup>'
    )
)

top_20 = centrality_df.head(20)

fig.add_trace(
    go.Bar(
        x=top_20['File'],
        y=top_20['Betweenness'],
        marker_color='indianred',
        name='Betweenness'
    ),
    row=1, col=1
)

# Sort by in-degree for second chart
top_20_indeg = centrality_df.nlargest(20, 'In-Degree')
fig.add_trace(
    go.Bar(
        x=top_20_indeg['File'],
        y=top_20_indeg['In-Degree'],
        marker_color='steelblue',
        name='In-Degree'
    ),
    row=1, col=2
)

# Sort by PageRank for third chart
top_20_pr = centrality_df.nlargest(20, 'PageRank')
fig.add_trace(
    go.Bar(
        x=top_20_pr['File'],
        y=top_20_pr['PageRank'],
        marker_color='darkgreen',
        name='PageRank'
    ),
    row=1, col=3
)

fig.update_xaxes(tickangle=-45, tickfont=dict(size=8))
fig.update_layout(
    height=500,
    showlegend=False,
    title_text='Centrality Analysis: Identifying Critical Files'
)
fig.show()

# Print summary
print(f"{'='*60}")
print("CENTRALITY ANALYSIS RESULTS")
print(f"{'='*60}")

print(f"\nTop 5 architectural bridges (betweenness centrality):")
print("  These files connect different parts of the system")
for idx, row in centrality_df.head(5).iterrows():
    print(f"    {row['File']:25s}  {row['Betweenness']:.4f}")

print(f"\nTop 5 most depended-upon files (in-degree):")
print("  Many other files include/depend on these")
for idx, row in centrality_df.nlargest(5, 'In-Degree').iterrows():
    print(f"    {row['File']:25s}  ← {int(row['In-Degree']):2d} files")

print(f"\nTop 5 by PageRank (overall importance):")
for idx, row in centrality_df.nlargest(5, 'PageRank').iterrows():
    print(f"    {row['File']:25s}  {row['PageRank']:.4f}")

print(f"\n✓ SYSTEM.H and PRISM.H emerge as critical infrastructure files")

### RQ2 Visualization 4: Community Detection

Machine learning-based module discovery using clustering algorithms.

In [ ]:
# =============================================================================
# RQ2 Visualization 4: Community Detection
# Uses: G, G_undirected, pos, degree_dict, betweenness (from Step 1)
# =============================================================================

from networkx.algorithms import community

print("Detecting natural module groupings...")
print()

# Use Louvain method for community detection (greedy modularity)
try:
    communities = community.greedy_modularity_communities(G_undirected)

    # Create community mapping
    community_map = {}
    for idx, comm in enumerate(communities):
        for node in comm:
            community_map[node] = idx

    modularity_Q = community.modularity(G_undirected, communities, weight=None)
    print(f"✓ Detected {len(communities)} natural module groupings "
          f"(modularity Q = {modularity_Q:.3f})")
    print(f"\nModule sizes:")
    for idx, comm in enumerate(communities):
        # Get representative files (top 3 by in-degree)
        comm_files = list(comm)
        comm_files_sorted = sorted(comm_files, key=lambda x: in_degree_dict.get(x, 0), reverse=True)
        top_files = ', '.join(comm_files_sorted[:3])
        print(f"  Module {idx+1}: {len(comm):2d} files  (e.g., {top_files})")

    # Prepare edge trace for visualisation
    edge_x = []
    edge_y = []
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=0.5, color='rgba(150,150,150,0.3)'),
        hoverinfo='none',
        mode='lines'
    )

    # Prepare node data
    node_x = []
    node_y = []
    node_text = []
    node_size = []
    node_colors = []
    node_labels = []

    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)

        degree = degree_dict.get(node, 0)
        comm_id = community_map.get(node, -1)
        in_deg = in_degree_dict.get(node, 0)

        hover = (f"<b>{node}</b><br>"
                 f"Module: {comm_id + 1}<br>"
                 f"Connections: {degree}<br>"
                 f"Incoming: {in_deg}")
        node_text.append(hover)
        node_size.append(max(8, degree * 1.5))
        node_colors.append(comm_id)

        label = node.replace('.C', '').replace('.H', '').replace('exec\\', '').replace('librarie\\', '')
        node_labels.append(label[:10])

    # Create visualisation
    fig = go.Figure(data=[
        edge_trace,
        go.Scatter(
            x=node_x, y=node_y,
            mode='markers+text',
            hoverinfo='text',
            hovertext=node_text,
            text=node_labels,
            textposition='top center',
            textfont=dict(size=7, color='#444444'),
            marker=dict(
                size=node_size,
                color=node_colors,
                colorscale='Viridis',
                showscale=True,
                colorbar=dict(
                    title='Module',
                    thickness=15,
                    len=0.5
                ),
                line=dict(width=1, color='white')
            )
        )
    ])

    fig.update_layout(
        title=dict(
            text='Community Detection: Natural Module Boundaries<br><sup>Colours show discovered functional modules (no prior labelling)</sup>',
            x=0.5,
            xanchor='center'
        ),
        showlegend=False,
        hovermode='closest',
        margin=dict(b=20, l=20, r=20, t=60),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=700,
        paper_bgcolor='white',
        plot_bgcolor='white'
    )

    fig.show()

    # Analyse module composition
    print(f"\n{'='*60}")
    print("MODULE COMPOSITION ANALYSIS")
    print(f"{'='*60}")

    for idx, comm in enumerate(communities[:5]):  # Top 5 modules
        comm_files = list(comm)
        # Categorise by file type
        headers = [f for f in comm_files if '.H' in f.upper()]
        sources = [f for f in comm_files if '.C' in f.upper()]

        print(f"\nModule {idx+1} ({len(comm)} files):")
        print(f"  Headers: {len(headers)}, Sources: {len(sources)}")

        # Show top files by importance
        top_by_between = sorted(comm_files, key=lambda x: betweenness.get(x, 0), reverse=True)[:3]
        print(f"  Key files: {', '.join(top_by_between)}")

except Exception as e:
    print(f"Community detection error: {e}")
    print("Falling back to naming convention analysis...")

    # Fallback: use Module column from file_complexity
    module_sizes = file_complexity['Module'].value_counts().head(10)
    fig = px.bar(
        module_sizes,
        title='Module Groupings from Naming Conventions',
        labels={'value': 'Number of Files', 'index': 'Module Prefix'}
    )
    fig.update_layout(height=500)
    fig.show()

    print(f"✓ {len(file_complexity['Module'].unique())} modules identified from naming patterns")

### RQ2 Visualization 5: Module Interaction Heatmap

Visualization of coupling between functional modules.

In [ ]:
# =============================================================================
# RQ2 Visualization 5: Module Interaction Heatmap
# Uses: file_dependencies, file_complexity (actual edge data)
# =============================================================================

print("Building module interaction heatmap from actual dependencies...")
print()

# Filter out external system dependencies (Amiga OS headers)
# These have paths like exec\types.h or librarie\dos.h
internal_deps = file_dependencies[
    ~file_dependencies['Target'].str.contains(r'\\|/', regex=True)
].copy()

print(f"Total dependencies: {len(file_dependencies)}")
print(f"Internal dependencies (Deluxe Paint only): {len(internal_deps)}")
print(f"External dependencies filtered out: {len(file_dependencies) - len(internal_deps)}")
print()

# Create module mapping from file_complexity (already computed)
# This ensures consistency with other analyses
module_map = dict(zip(file_complexity['Name'], file_complexity['Module']))

# Function to get module, with fallback for files not in file_complexity
def get_module(filename):
    """Get module for a filename, using existing mapping or extracting prefix."""
    if filename in module_map:
        return module_map[filename]
    # Fallback: extract leading uppercase letters
    import re
    match = re.match(r'^([A-Z]+)', filename.upper())
    if match:
        return match.group(1)
    return 'OTHER'

# Add module columns
internal_deps['SourceModule'] = internal_deps['Source'].apply(get_module)
internal_deps['TargetModule'] = internal_deps['Target'].apply(get_module)

# Get top modules by total interaction weight
module_weights = pd.concat([
    internal_deps.groupby('SourceModule')['weight'].sum(),
    internal_deps.groupby('TargetModule')['weight'].sum()
]).groupby(level=0).sum().sort_values(ascending=False)

top_modules = module_weights.head(12).index.tolist()

# Build actual interaction matrix from edge data
interaction_matrix = pd.DataFrame(0.0, index=top_modules, columns=top_modules)

for idx, row in internal_deps.iterrows():
    src_mod = row['SourceModule']
    tgt_mod = row['TargetModule']
    weight = row['weight']

    if src_mod in top_modules and tgt_mod in top_modules:
        interaction_matrix.loc[src_mod, tgt_mod] += weight

# Create heatmap
fig = px.imshow(
    interaction_matrix,
    labels=dict(x="Target Module", y="Source Module", color="Dependency<br>Weight"),
    x=interaction_matrix.columns,
    y=interaction_matrix.index,
    color_continuous_scale='Reds',
    aspect='auto'
)

fig.update_layout(
    title=dict(
        text='Module Interaction Heatmap<br><sup>Actual dependency weights between Deluxe Paint modules</sup>',
        x=0.5,
        xanchor='center'
    ),
    height=600
)

fig.show()

# Print summary statistics
print(f"{'='*60}")
print("MODULE INTERACTION ANALYSIS")
print(f"{'='*60}")

# Find strongest cross-module dependencies
cross_module = []
for src in top_modules:
    for tgt in top_modules:
        if src != tgt and interaction_matrix.loc[src, tgt] > 0:
            cross_module.append((src, tgt, interaction_matrix.loc[src, tgt]))

cross_module = sorted(cross_module, key=lambda x: x[2], reverse=True)

print(f"\nTop 10 cross-module dependencies:")
for src, tgt, weight in cross_module[:10]:
    print(f"  {src:12s} → {tgt:12s}  weight: {weight:.0f}")

# Internal cohesion (diagonal values)
print(f"\nModule internal cohesion (diagonal):")
for mod in top_modules[:5]:
    internal = interaction_matrix.loc[mod, mod]
    print(f"  {mod:12s}  internal weight: {internal:.0f}")

# Most depended-upon modules (column sums, excluding diagonal)
incoming = interaction_matrix.sum(axis=0) - pd.Series({m: interaction_matrix.loc[m, m] for m in top_modules})
incoming = incoming.sort_values(ascending=False)
print(f"\nMost depended-upon modules (incoming from other modules):")
for mod, weight in incoming.head(5).items():
    print(f"  {mod:12s}  ← {weight:.0f}")

print(f"\n✓ Heatmap shows actual coupling between functional modules")
print(f"  Diagonal = internal dependencies within module")
print(f"  Off-diagonal = cross-module dependencies")

### RQ2 Visualization 6: Network Metrics Dashboard



In [ ]:
# =============================================================================
# RQ2 Visualization 6: Network Metrics Dashboard
# Uses: G, G_undirected, degree_dict, in_degree_dict, out_degree_dict (from Step 1)
# =============================================================================

print("Computing network health metrics...")
print()

# Basic metrics (using pre-built graph)
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
density = nx.density(G)

# Connectivity metrics
try:
    avg_clustering = nx.average_clustering(G_undirected)
except:
    avg_clustering = 0

# Degree statistics (using pre-computed dictionaries)
degree_values = list(degree_dict.values())
in_degree_values = list(in_degree_dict.values())
out_degree_values = list(out_degree_dict.values())

max_degree = max(degree_values) if degree_values else 0
avg_degree = sum(degree_values) / len(degree_values) if degree_values else 0
max_in_degree = max(in_degree_values) if in_degree_values else 0
max_out_degree = max(out_degree_values) if out_degree_values else 0

# Path lengths (on largest connected component)
if nx.is_connected(G_undirected):
    avg_path_length = nx.average_shortest_path_length(G_undirected)
    diameter = nx.diameter(G_undirected)
else:
    largest_cc = max(nx.connected_components(G_undirected), key=len)
    subgraph = G_undirected.subgraph(largest_cc)
    avg_path_length = nx.average_shortest_path_length(subgraph)
    diameter = nx.diameter(subgraph)

# Find the most connected file
most_connected = max(degree_dict.items(), key=lambda x: x[1])
most_depended = max(in_degree_dict.items(), key=lambda x: x[1])

# Create visual dashboard
fig = make_subplots(
    rows=2, cols=3,
    specs=[
        [{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}],
        [{'type': 'bar', 'colspan': 2}, None, {'type': 'pie'}]
    ],
    subplot_titles=('', '', '', 'Degree Distribution', '', 'In vs Out Degree')
)

# Row 1: Key indicators
fig.add_trace(go.Indicator(
    mode="number",
    value=num_nodes,
    title={'text': "Files in Network", 'font': {'size': 14}},
    number={'font': {'size': 40}}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number",
    value=num_edges,
    title={'text': "Dependencies", 'font': {'size': 14}},
    number={'font': {'size': 40}}
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode="number",
    value=round(avg_path_length, 2),
    title={'text': "Avg Path Length", 'font': {'size': 14}},
    number={'font': {'size': 40}}
), row=1, col=3)

# Row 2: Degree distribution histogram
degree_dist = pd.Series(degree_values).value_counts().sort_index()
fig.add_trace(go.Bar(
    x=degree_dist.index,
    y=degree_dist.values,
    marker_color='steelblue',
    name='Degree'
), row=2, col=1)

# Row 2: In-degree vs Out-degree pie
total_in = sum(in_degree_values)
total_out = sum(out_degree_values)
fig.add_trace(go.Pie(
    labels=['Incoming', 'Outgoing'],
    values=[total_in, total_out],
    marker_colors=['#2ecc71', '#e74c3c'],
    hole=0.4
), row=2, col=3)

fig.update_xaxes(title_text="Degree (Connections)", row=2, col=1)
fig.update_yaxes(title_text="Number of Files", row=2, col=1)

fig.update_layout(
    height=550,
    showlegend=False,
    title_text='Network Health Metrics Dashboard'
)
fig.show()

# Print detailed metrics table
print(f"{'='*60}")
print("NETWORK METRICS SUMMARY")
print(f"{'='*60}")

metrics = [
    ("Nodes (files)", num_nodes, "Total files in dependency graph"),
    ("Edges (dependencies)", num_edges, "Total dependency relationships"),
    ("Density", f"{density:.4f}", "Edge count / possible edges"),
    ("Avg clustering", f"{avg_clustering:.3f}", "How files cluster together"),
    ("Avg path length", f"{avg_path_length:.2f}", "Avg steps between any two files"),
    ("Diameter", diameter, "Maximum shortest path"),
    ("Max degree", f"{max_degree} ({most_connected[0]})", "Most connected file"),
    ("Max in-degree", f"{max_in_degree} ({most_depended[0]})", "Most depended-upon file"),
    ("Avg degree", f"{avg_degree:.1f}", "Average connections per file"),
]

for metric, value, description in metrics:
    print(f"  {metric:20s}  {str(value):25s}  {description}")

# Interpretation
print(f"\n{'='*60}")
print("INTERPRETATION")
print(f"{'='*60}")

if density < 0.1:
    density_interp = "loosely connected (modular design)"
elif density < 0.3:
    density_interp = "moderately connected"
else:
    density_interp = "highly connected (potential coupling issues)"
print(f"  Density {density:.4f}: {density_interp}")

if avg_path_length < 3:
    path_interp = "compact architecture (easy navigation)"
elif avg_path_length < 5:
    path_interp = "moderate architecture"
else:
    path_interp = "distributed architecture (potential coordination overhead)"
print(f"  Path length {avg_path_length:.2f}: {path_interp}")

if avg_clustering > 0.3:
    cluster_interp = "strong clustering (clear module boundaries)"
else:
    cluster_interp = "weak clustering (files interact across modules)"
print(f"  Clustering {avg_clustering:.3f}: {cluster_interp}")

print(f"\n✓ Most critical file: {most_depended[0]} ({most_depended[1]} files depend on it)")

### RQ2 Visualization 7: Network Robustness

Simulation of system failure if key hub files were removed.

In [ ]:
# =============================================================================
# RQ2 Visualization 7: Network Robustness
# Uses: G, G_undirected, betweenness, in_degree_dict (from Step 1)
# =============================================================================

print("Simulating architectural robustness...")
print("What happens if critical files are removed?")
print()

# We'll test two removal strategies:
# 1. Remove by betweenness centrality (architectural bridges)
# 2. Remove by in-degree (most depended-upon files)

sorted_by_betweenness = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)
sorted_by_indegree = sorted(in_degree_dict.items(), key=lambda x: x[1], reverse=True)

def simulate_removal(G_original, removal_order, max_removals=10):
    """Simulate progressive node removal and track network fragmentation."""
    G_test = G_original.to_undirected().copy()
    initial_nodes = G_test.number_of_nodes()
    results = []

    for i in range(min(max_removals, len(removal_order))):
        node_to_remove = removal_order[i][0]

        if node_to_remove in G_test:
            G_test.remove_node(node_to_remove)

            if G_test.number_of_nodes() > 0:
                num_components = nx.number_connected_components(G_test)
                largest_cc = max(nx.connected_components(G_test), key=len)
                largest_size = len(largest_cc)
                connectivity = largest_size / initial_nodes  # fraction of the ORIGINAL graph
            else:
                num_components = 0
                largest_size = 0
                connectivity = 0

            results.append({
                'Removed': i + 1,
                'File': node_to_remove,
                'Components': num_components,
                'Largest': largest_size,
                'Connectivity': connectivity,
                'Remaining': G_test.number_of_nodes()
            })

    return pd.DataFrame(results)

# Run simulations
results_betweenness = simulate_removal(G, sorted_by_betweenness)
results_indegree = simulate_removal(G, sorted_by_indegree)

# Create comparison visualisation
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Network Fragmentation (Components)',
        'Connectivity Loss (Largest Component %)',
        'Files Removed by Betweenness',
        'Files Removed by In-Degree'
    ),
    specs=[
        [{'type': 'scatter'}, {'type': 'scatter'}],
        [{'type': 'table'}, {'type': 'table'}]
    ],
    row_heights=[0.6, 0.4]
)

# Plot 1: Fragmentation comparison
fig.add_trace(go.Scatter(
    x=results_betweenness['Removed'],
    y=results_betweenness['Components'],
    mode='lines+markers',
    marker=dict(size=8, color='red'),
    line=dict(color='red', width=2),
    name='By Betweenness'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=results_indegree['Removed'],
    y=results_indegree['Components'],
    mode='lines+markers',
    marker=dict(size=8, color='blue'),
    line=dict(color='blue', width=2, dash='dash'),
    name='By In-Degree'
), row=1, col=1)

# Plot 2: Connectivity comparison
fig.add_trace(go.Scatter(
    x=results_betweenness['Removed'],
    y=results_betweenness['Connectivity'] * 100,
    mode='lines+markers',
    marker=dict(size=8, color='red'),
    line=dict(color='red', width=2),
    name='By Betweenness',
    showlegend=False
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=results_indegree['Removed'],
    y=results_indegree['Connectivity'] * 100,
    mode='lines+markers',
    marker=dict(size=8, color='blue'),
    line=dict(color='blue', width=2, dash='dash'),
    name='By In-Degree',
    showlegend=False
), row=1, col=2)

# Table 1: Betweenness removal order
fig.add_trace(go.Table(
    header=dict(
        values=['#', 'File', 'Components', 'Connectivity'],
        fill_color='indianred',
        font=dict(color='white', size=10),
        align='left'
    ),
    cells=dict(
        values=[
            results_betweenness['Removed'],
            results_betweenness['File'],
            results_betweenness['Components'],
            [f"{v:.0%}" for v in results_betweenness['Connectivity']]
        ],
        fill_color='lavenderblush',
        align='left',
        font=dict(size=9)
    )
), row=2, col=1)

# Table 2: In-degree removal order
fig.add_trace(go.Table(
    header=dict(
        values=['#', 'File', 'Components', 'Connectivity'],
        fill_color='steelblue',
        font=dict(color='white', size=10),
        align='left'
    ),
    cells=dict(
        values=[
            results_indegree['Removed'],
            results_indegree['File'],
            results_indegree['Components'],
            [f"{v:.0%}" for v in results_indegree['Connectivity']]
        ],
        fill_color='lavender',
        align='left',
        font=dict(size=9)
    )
), row=2, col=2)

fig.update_xaxes(title_text="Files Removed", row=1, col=1)
fig.update_xaxes(title_text="Files Removed", row=1, col=2)
fig.update_yaxes(title_text="Components", row=1, col=1)
fig.update_yaxes(title_text="Connectivity %", row=1, col=2)

fig.update_layout(
    height=700,
    title_text='Network Robustness: Hub File Failure Simulation',
    legend=dict(x=0.02, y=0.98)
)
fig.show()

# Print analysis
print(f"{'='*60}")
print("ROBUSTNESS ANALYSIS")
print(f"{'='*60}")

print(f"\nRemoving top 10 files by BETWEENNESS (architectural bridges):")
print(f"  Network fragments into {results_betweenness.iloc[-1]['Components']} components")
print(f"  Connectivity drops to {results_betweenness.iloc[-1]['Connectivity']:.1%}")

print(f"\nRemoving top 10 files by IN-DEGREE (most depended-upon):")
print(f"  Network fragments into {results_indegree.iloc[-1]['Components']} components")
print(f"  Connectivity drops to {results_indegree.iloc[-1]['Connectivity']:.1%}")

# Determine which strategy causes more damage
if results_betweenness.iloc[-1]['Components'] > results_indegree.iloc[-1]['Components']:
    print(f"\n⚠ Removing BRIDGE files causes more fragmentation")
    print(f"  These files connect otherwise separate parts of the system")
else:
    print(f"\n⚠ Removing HIGHLY-DEPENDED files causes more fragmentation")
    print(f"  These are foundational files (like SYSTEM.H, PRISM.H)")

print(f"\n{'='*60}")
print("ARCHITECTURAL IMPLICATIONS")
print(f"{'='*60}")
print(f"  ✓ Hub-and-spoke design provides centralised coordination")
print(f"  ✗ Critical files are single points of failure")
print(f"  → SYSTEM.H and PRISM.H are essential infrastructure")
print(f"  → Removing them would require significant refactoring")

In [ ]:
# =============================================================================
# RQ2 Visualization: Module Interaction Flow (Sankey Diagram)
# Uses: file_dependencies (edge data)
# =============================================================================

import plotly.graph_objects as go

print("Building module interaction flow diagram...")
print()

# Filter out external system dependencies (Amiga OS headers)
internal_deps = file_dependencies[
    ~file_dependencies['Target'].str.contains(r'\\|/', regex=True)
].copy()

# Extract module names (file name without extension)
def get_module_name(filename):
    """Extract module name from filename (e.g., PRISM.C -> PRISM)."""
    name = filename.split('\\')[-1].split('/')[-1]  # Remove path
    name = name.replace('.C', '').replace('.H', '').replace('.c', '').replace('.h', '')
    return name

internal_deps['SourceModule'] = internal_deps['Source'].apply(get_module_name)
internal_deps['TargetModule'] = internal_deps['Target'].apply(get_module_name)

# Aggregate dependencies between modules
module_flows = internal_deps.groupby(['SourceModule', 'TargetModule'])['weight'].sum().reset_index()

# Filter out self-references and very weak connections
module_flows = module_flows[module_flows['SourceModule'] != module_flows['TargetModule']]
module_flows = module_flows[module_flows['weight'] >= 1]

# Get all unique modules
all_modules = list(set(module_flows['SourceModule'].tolist() + module_flows['TargetModule'].tolist()))
all_modules = sorted(all_modules)

# Create node indices
node_indices = {module: idx for idx, module in enumerate(all_modules)}

# Prepare Sankey data
sources = [node_indices[src] for src in module_flows['SourceModule']]
targets = [node_indices[tgt] for tgt in module_flows['TargetModule']]
values = module_flows['weight'].tolist()

# Calculate node positions (optional: group by dependency direction)
# Nodes with more outgoing deps on left, more incoming on right
outgoing = module_flows.groupby('SourceModule')['weight'].sum()
incoming = module_flows.groupby('TargetModule')['weight'].sum()

# Determine x position based on ratio of outgoing to incoming
node_x = []
node_y = []
for module in all_modules:
    out_w = outgoing.get(module, 0)
    in_w = incoming.get(module, 0)
    total = out_w + in_w
    if total > 0:
        # More outgoing = closer to left (0), more incoming = closer to right (1)
        x_pos = in_w / total
    else:
        x_pos = 0.5
    node_x.append(x_pos)

# Create Sankey diagram
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color='white', width=0.5),
        label=all_modules,
        color='rgba(31, 119, 180, 0.8)',
        x=node_x,
        hovertemplate='%{label}<br>Total flow: %{value}<extra></extra>'
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color='rgba(31, 119, 180, 0.2)',
        hovertemplate='%{source.label} → %{target.label}<br>Weight: %{value}<extra></extra>'
    )
)])

fig.update_layout(
    title=dict(
        text='Module Interaction Flow<br><sup>Dependency flow between major code modules</sup>',
        x=0.5,
        xanchor='center'
    ),
    font=dict(size=10),
    height=700,
    paper_bgcolor='white',
    annotations=[
        dict(
            x=0.5, y=-0.12,
            xref='paper', yref='paper',
            text='<b>How to read this diagram:</b><br>'
                 '• <b>Nodes:</b> Code modules (identified by file prefixes)<br>'
                 '• <b>Flows:</b> Dependencies from source to target module<br>'
                 '• <b>Width:</b> Number/strength of dependencies<br>'
                 '• <b>Pattern:</b> Wide flows indicate tight coupling between modules',
            showarrow=False,
            font=dict(size=11),
            align='center',
            bgcolor='rgba(240,245,250,0.9)',
            bordercolor='#cccccc',
            borderwidth=1,
            borderpad=8
        )
    ],
    margin=dict(b=120)
)

fig.show()

# Print summary
print(f"{'='*60}")
print("MODULE FLOW SUMMARY")
print(f"{'='*60}")

# Top dependency flows
top_flows = module_flows.nlargest(10, 'weight')
print(f"\nTop 10 strongest dependency flows:")
for idx, row in top_flows.iterrows():
    print(f"  {row['SourceModule']:15s} → {row['TargetModule']:15s}  weight: {row['weight']:.0f}")

# Most connected modules (by total flow)
total_flow = pd.concat([
    module_flows.groupby('SourceModule')['weight'].sum().rename('Outgoing'),
    module_flows.groupby('TargetModule')['weight'].sum().rename('Incoming')
], axis=1).fillna(0)
total_flow['Total'] = total_flow['Outgoing'] + total_flow['Incoming']
total_flow = total_flow.sort_values('Total', ascending=False)

print(f"\nTop 10 modules by total dependency flow:")
print(f"  {'Module':<15s} {'Outgoing':>10s} {'Incoming':>10s} {'Total':>10s}")
print(f"  {'-'*45}")
for module in total_flow.head(10).index:
    row = total_flow.loc[module]
    print(f"  {module:<15s} {row['Outgoing']:>10.0f} {row['Incoming']:>10.0f} {row['Total']:>10.0f}")

print(f"\n✓ PRISM and SYSTEM modules are central hubs receiving most dependencies")

---
## RQ4: Coding idioms

**What coding idioms are present in the source code of *Deluxe Paint*?**

The Amiga 1000 had 256 KB of RAM, split into Chip RAM (shared with the custom chips) and Fast RAM (CPU only), with no virtual memory and no garbage collection. We examine the source-level idioms that respond to these constraints: code, comment, and preprocessor density, function parameter use, and coupling and instability.

---


In [ ]:
# RQ4: Memory-Conscious Design Analysis
print("RQ4: MEMORY-CONSCIOUS PROGRAMMING STRATEGIES")
print("="*80)
print("Analyzing code characteristics that indicate memory optimization...\n")

# Code density metrics (executable code vs total)
# 'Executable lines' = CountLineCodeExe, which excludes declaration-only lines.
file_complexity['CodeDensity'] = file_complexity['CountLineCodeExe'] / file_complexity['CountLine']
file_complexity['CommentDensity'] = file_complexity['CountLineComment'] / file_complexity['CountLine']

print("Code Density Analysis:")
print(f"  Average Code Density: {file_complexity['CodeDensity'].mean():.1%}")
print(f"  Average Comment Density: {file_complexity['CommentDensity'].mean():.1%}")
print(f"  Average Blank Line Ratio: {(file_complexity['CountLineBlank'] / file_complexity['CountLine']).mean():.1%}")
print(f"  Average Preprocessor Density: {(file_complexity['CountLinePreprocessor'] / file_complexity['CountLine']).mean():.1%}")

print("\nInterpretation:")
print(f"  Executable lines are ~{file_complexity['CodeDensity'].mean():.0%} of each file on average;")
print(f"  comments are ~{file_complexity['CommentDensity'].mean():.0%}. The remainder is declarations,")
print("  blank lines, and preprocessor directives (compact, single-developer C).")

# Function parameter analysis (memory-conscious function signatures)
print(f"\nFunction Parameter Analysis:")
print(f"  Average parameters per function: {function_complexity['CountInput'].mean():.1f}")
print(f"  Functions with 0-2 parameters: {len(function_complexity[function_complexity['CountInput'] <= 2])} ({len(function_complexity[function_complexity['CountInput'] <= 2])/len(function_complexity)*100:.1f}%)")
print(f"  Functions with >5 parameters: {len(function_complexity[function_complexity['CountInput'] > 5])}")

if function_complexity['CountInput'].mean() < 4:
    print("\n  ✓ LOW average parameter count suggests:")
    print("    - Use of global state (reduced stack usage)")
    print("    - Pointer passing instead of value copying")
    print("    - Memory-conscious function design")

# File size distribution (how they managed module sizes)
print(f"\nFile Size Distribution:")
print(f"  Files < 200 LOC: {len(file_complexity[file_complexity['CountLine'] < 200])}")
print(f"  Files 200-500 LOC: {len(file_complexity[(file_complexity['CountLine'] >= 200) & (file_complexity['CountLine'] < 500)])}")
print(f"  Files > 500 LOC: {len(file_complexity[file_complexity['CountLine'] > 500])}")
print(f"  Largest file: {file_complexity['CountLine'].max():.0f} LOC ({file_complexity.loc[file_complexity['CountLine'].idxmax(), 'Name']})")

print("\n  ✓ Majority of files are MODULAR and COMPACT")
print("    Smaller files = faster compilation on slow hardware")

In [ ]:
# RQ4: Efficiency Indicators
print("\nRQ4: CODE EFFICIENCY INDICATORS")
print("="*80)

# Calculate efficiency metrics
file_complexity['ComplexityPerLOC'] = file_complexity['SumCyclomatic'] / file_complexity['CountLine']
file_complexity['FunctionDensity'] = file_complexity['CountDeclFunction'] / file_complexity['CountLine'] * 100

print("\nComplexity Efficiency (how much logic per line):")
print(f"  Average Complexity per LOC: {file_complexity['ComplexityPerLOC'].mean():.3f}")
print(f"  This indicates: {'HIGH' if file_complexity['ComplexityPerLOC'].mean() > 0.1 else 'MODERATE'} logic density")

print("\nFunction Density (functions per 100 LOC):")
print(f"  Average: {file_complexity['FunctionDensity'].mean():.1f} functions per 100 LOC")

# Preprocessor usage (macros for efficiency)
print(f"\nPreprocessor Usage:")
print(f"  Average preprocessor directives per file: {file_complexity['CountLinePreprocessor'].mean():.1f}")
print(f"  Files with >20 preprocessor lines: {len(file_complexity[file_complexity['CountLinePreprocessor'] > 20])}")

if file_complexity['CountLinePreprocessor'].mean() > 10:
    print("\n  ✓ HEAVY use of preprocessor (macros, conditional compilation)")
    print("    Macros = inline expansion (no function call overhead)")
    print("    Critical for performance in 7.16 MHz CPU")

# Visualize efficiency trade-offs
fig = px.scatter(
    file_complexity,
    x='CodeDensity',
    y='ComplexityPerLOC',
    size='CountLine',
    color='RatioCommentToCode',
    hover_data=['Name'],
    title='RQ4: Code Density vs Complexity Efficiency<br><sub>Trade-offs in memory-constrained development</sub>',
    labels={
        'CodeDensity': 'Code Density (executable LOC / total LOC)',
        'ComplexityPerLOC': 'Complexity per Line of Code',
        'RatioCommentToCode': 'Comment Ratio'
    },
    color_continuous_scale='Viridis'
)
fig.update_layout(height=600)
fig.show()

print("\nInterpretation: High density + high complexity/LOC = OPTIMIZED CODE")
print("Developers packed maximum functionality into minimum space")

In [ ]:
# RQ4: Documentation under memory constraints.
# We use comment density (comment lines / total lines) rather than the comment-to-code ratio.
print("\nRQ4: DOCUMENTATION UNDER MEMORY CONSTRAINTS")
print("="*80)
print("Did memory constraints sacrifice documentation?\n")

comment_density = file_complexity['CountLineComment'] / file_complexity['CountLine']
mean_density = comment_density.mean()
modern_norm = 0.25  # commonly cited 20-30% guideline

print("Comment density (comment lines / total lines):")
print(f"  Mean comment density:            {mean_density:.2f}")
print(f"  Commonly cited modern guideline: ~{modern_norm:.2f}")

dense = int((comment_density > 0.30).sum())
sparse = int((comment_density < 0.10).sum())
hi_name = file_complexity.loc[comment_density.idxmax(), 'Name']
print(f"\n  Densely commented (> 0.30):  {dense} files")
print(f"  Sparsely commented (< 0.10): {sparse} files")
print(f"  Highest comment density: {comment_density.max():.2f} ({hi_name})")

print("\nInterpretation:")
print(f"  At a mean of {mean_density:.2f}, comment density is modest and below the")
print("  commonly cited 0.20-0.30 guideline. This is consistent with compact,")
print("  single-developer C in which the author is the primary reader. Comments are")
print("  stripped at compile time, so their sparseness is a source-level trait,")
print("  not a memory optimisation.")

corr_complex_doc = file_complexity['SumCyclomatic'].corr(comment_density)
print(f"\nCorrelation (file complexity vs comment density): {corr_complex_doc:.3f}")
print("  → No strong relationship between complexity and documentation.")

### RQ4 Analysis 3: Coupling and Cohesion Metrics

Measure modular design quality through coupling (dependencies) and instability metrics.

In [ ]:
# RQ4: Coupling and Cohesion Analysis
# Measure modular design quality
import pandas as pd

print("="*80)
print("RQ4: COUPLING AND COHESION ANALYSIS")
print("="*80)
print("Measuring modular design quality...\n")

# Calculate coupling metrics from dependency data
coupling_data = []
for idx, row in file_complexity.iterrows():
    filename = row['Name']
    # Efferent coupling (outgoing): how many files this file depends on
    ce = internal_dependencies[internal_dependencies['Source'] == filename]['Target'].nunique()
    # Afferent coupling (incoming): how many files depend on this file (internal graph)
    ca = internal_dependencies[internal_dependencies['Target'] == filename]['Source'].nunique()

    # Instability metric: I = Ce / (Ce + Ca)
    # I = 0 (stable, depended upon) to I = 1 (unstable, depends on others)
    total_coupling = ce + ca
    instability = ce / total_coupling if total_coupling > 0 else 0

    # Abstractness proxy: header files are more abstract
    is_header = filename.endswith('.H')
    coupling_data.append({
        'File': filename,
        'Efferent': ce,
        'Afferent': ca,
        'Total': total_coupling,
        'Instability': instability,
        'Type': 'Header' if is_header else 'Implementation'
    })

coupling_df = pd.DataFrame(coupling_data)

print(f"Coupling Metrics Summary:\n")
print(f"{'Metric':<25} {'Mean':<10} {'Median':<10} {'Max'}")
print("-"*55)
print(f"{'Efferent Coupling (Ce)':<25} {coupling_df['Efferent'].mean():<10.1f} {coupling_df['Efferent'].median():<10.1f} {coupling_df['Efferent'].max()}")
print(f"{'Afferent Coupling (Ca)':<25} {coupling_df['Afferent'].mean():<10.1f} {coupling_df['Afferent'].median():<10.1f} {coupling_df['Afferent'].max()}")
print(f"{'Instability (I)':<25} {coupling_df['Instability'].mean():<10.3f} {coupling_df['Instability'].median():<10.3f} {coupling_df['Instability'].max():.3f}")

# Stable vs Unstable files
stable = coupling_df[coupling_df['Instability'] < 0.3]
unstable = coupling_df[coupling_df['Instability'] > 0.7]

print(f"\nStability Distribution:")
print(f"  Stable files (I < 0.3):   {len(stable):>3d} ({len(stable)/len(coupling_df)*100:>5.1f}%)")
print(f"  Balanced files (0.3-0.7): {len(coupling_df[(coupling_df['Instability'] >= 0.3) & (coupling_df['Instability'] <= 0.7)]):>3d}")
print(f"  Unstable files (I > 0.7): {len(unstable):>3d} ({len(unstable)/len(coupling_df)*100:>5.1f}%)")

# Most depended-upon files (stable, reusable components)
print(f"\nTop 10 Most Depended-Upon Files (Stable Components):")
print(f"{'File':<20} {'Afferent':<10} {'Efferent':<10} {'Instability'}")
print("-"*55)
for idx, row in coupling_df.nlargest(10, 'Afferent').iterrows():
    print(f"{row['File']:<20} {row['Afferent']:<10} {row['Efferent']:<10} {row['Instability']:.3f}")

print(f"\n✓ Clear separation between stable (depended-upon) and unstable (dependent) files")
print(f"✓ Headers act as stable interfaces (low instability)")

---
## RQ5: Complexity distribution

**How was complexity distributed across the different components of *Deluxe Paint*, and why?**

Graphics software involves algorithmically complex operations: bitmap transformations, colour-palette manipulation, real-time rendering with the hardware blitter, and user input handling. We examine how cyclomatic complexity is distributed across files and functions, by functional category, and in relation to file size and nesting depth.

---


In [ ]:
# RQ5: Complexity Distribution Analysis
print("RQ5: COMPLEXITY DISTRIBUTION ANALYSIS")
print("="*80)
print("Analyzing how complexity is distributed across the graphics application...\n")

# File-level complexity statistics
print("File-Level Cyclomatic Complexity:")
print(f"  Mean: {file_complexity['SumCyclomatic'].mean():.1f}")
print(f"  Median: {file_complexity['SumCyclomatic'].median():.1f}")
print(f"  Std Dev: {file_complexity['SumCyclomatic'].std():.1f}")
print(f"  Max: {file_complexity['SumCyclomatic'].max():.0f} ({file_complexity.loc[file_complexity['SumCyclomatic'].idxmax(), 'Name']})")

# Function-level complexity distribution
print(f"\nFunction-Level Cyclomatic Complexity:")
print(f"  Mean: {function_complexity['Cyclomatic'].mean():.1f}")
print(f"  Median: {function_complexity['Cyclomatic'].median():.1f}")
print(f"  Std Dev: {function_complexity['Cyclomatic'].std():.1f}")
print(f"  Max: {function_complexity['Cyclomatic'].max():.0f} ({function_complexity.loc[function_complexity['Cyclomatic'].idxmax(), 'Name']})")
print(f"  Functions with CC <= 5: {(function_complexity['Cyclomatic'] <= 5).mean()*100:.0f}%")

# Complexity risk distribution
print(f"\nComplexity Risk Distribution (McCabe's Thresholds):")
risk_dist = function_complexity['ComplexityRisk'].value_counts()
for risk, count in risk_dist.items():
    pct = count / len(function_complexity) * 100
    print(f"  {risk:15s}: {count:3d} functions ({pct:5.1f}%)")

# High complexity functions
high_complexity = function_complexity[function_complexity['Cyclomatic'] > 15]
print(f"\n⚠️  High Complexity Functions (CC > 15): {len(high_complexity)}")
print("\nTop 10 Most Complex Functions:")
print(high_complexity.nlargest(10, 'Cyclomatic')[['Name', 'Cyclomatic', 'CountLine', 'MaxNesting']])

In [ ]:
# RQ5: Complexity Distribution Visualizations
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'File-Level Complexity Distribution',
        'Function-Level Complexity Distribution',
        'Complexity vs File Size',
        'Complexity Risk Categories'
    ),
    specs=[[{'type': 'histogram'}, {'type': 'histogram'}],
           [{'type': 'scatter'}, {'type': 'bar'}]]
)

# File complexity distribution
fig.add_trace(
    go.Histogram(x=file_complexity['SumCyclomatic'], nbinsx=30,
                 marker_color='lightblue', name='Files'),
    row=1, col=1
)

# Function complexity distribution
fig.add_trace(
    go.Histogram(x=function_complexity['Cyclomatic'], nbinsx=30,
                 marker_color='lightcoral', name='Functions'),
    row=1, col=2
)

# Complexity vs Size scatter
fig.add_trace(
    go.Scatter(x=file_complexity['CountLine'],
               y=file_complexity['SumCyclomatic'],
               mode='markers', marker=dict(size=8, color='green', opacity=0.6),
               name='Files'),
    row=2, col=1
)

# Risk categories bar chart
risk_counts = function_complexity['ComplexityRisk'].value_counts().sort_index()
colors = {'Low': 'green', 'Medium': 'yellow', 'High': 'orange', 'Very High': 'red'}
fig.add_trace(
    go.Bar(x=risk_counts.index, y=risk_counts.values,
           marker_color=[colors.get(x, 'gray') for x in risk_counts.index],
           name='Risk'),
    row=2, col=2
)

fig.update_xaxes(title_text="Cyclomatic Complexity", row=1, col=1)
fig.update_xaxes(title_text="Cyclomatic Complexity", row=1, col=2)
fig.update_xaxes(title_text="Lines of Code", row=2, col=1)
fig.update_xaxes(title_text="Complexity Risk", row=2, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_yaxes(title_text="Cyclomatic Complexity", row=2, col=1)
fig.update_yaxes(title_text="Number of Functions", row=2, col=2)

fig.update_layout(height=800, showlegend=False,
                  title_text="RQ5: Complexity Distribution Analysis")
fig.show()

In [ ]:
# RQ5 -- Complexity by functional category.
# We group the 77 .C source files into seven functional categories according to
# their primary purpose, and report how lines of code and cyclomatic complexity
# concentrate in each category.
print("RQ5: COMPLEXITY BY FUNCTIONAL CATEGORY")
print("=" * 80)


_order = ['UI / Interaction', 'System / Init', 'Bitmap / Blitter', 'File I/O',
          'Palette / Colour', 'Drawing / Graphics', 'Transforms']
_src = file_complexity[file_complexity['Name'].str.endswith('.C')]
category_stats = (_src.groupby('Category')
                  .agg(Files=('Name', 'count'),
                       LOC=('CountLine', 'sum'),
                       Sum_CC=('SumCyclomatic', 'sum'))
                  .reindex(_order))
category_stats['Avg_CC_per_file'] = (category_stats['Sum_CC'] / category_stats['Files']).round(1)

print(category_stats.to_string())
print(f"\n  Total: {int(category_stats.Files.sum())} files, "
      f"{int(category_stats.LOC.sum()):,} LOC, {int(category_stats.Sum_CC.sum()):,} cyclomatic complexity")
print("\nInterpretation: UI/interaction code carries the highest average complexity")
print("per file, driven by event-dispatch files such as CHPROC.C, MODES.C, and PANE.C.")


In [ ]:
# RQ5: Correlation Analysis - What drives complexity?
print("\nRQ5: FACTORS DRIVING COMPLEXITY")
print("="*80)
print("Examining relationships between complexity and other metrics...\n")

# Calculate correlations
metrics = ['CountLine', 'CountLineCode', 'CountDeclFunction', 'RatioCommentToCode']
target = 'SumCyclomatic'

_nz = file_complexity[file_complexity['SumCyclomatic'] > 0]  # files with non-zero complexity (N=56)
print(f"Correlation with File Cyclomatic Complexity (N={len(_nz)} files with CC>0):")
for metric in metrics:
    corr, pval = pearsonr(_nz[metric], _nz[target])
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
    print(f"  {metric:25s}: r = {corr:6.3f}  {sig}")

print("\n*** p<0.001, ** p<0.01, * p<0.05, ns=not significant")

# Nesting depth vs complexity (function level)
corr_nest, pval_nest = pearsonr(function_complexity['MaxNesting'],
                                 function_complexity['Cyclomatic'])
print(f"\nFunction Nesting Depth vs Cyclomatic Complexity: r = {corr_nest:.3f}")
print(f"P-value: {pval_nest:.4f}")

if corr_nest > 0.5:
    print("\n✓ Finding: Deep nesting STRONGLY correlates with complexity")
    print("  Hardware register manipulation requires nested conditionals")

### RQ5 Visualization 1: Complexity Heatmap

Visual representation of cyclomatic complexity across the codebase.

In [ ]:
# Create complexity heatmap
print("Building complexity heatmap...")

# Prepare data: organize files by module and sort by complexity
heatmap_data = file_complexity.copy()
heatmap_data = heatmap_data.sort_values(['Module', 'SumCyclomatic'], ascending=[True, False])

# Take top files from each major module
major_modules = heatmap_data['Module'].value_counts().head(8).index
viz_files = []
for module in major_modules:
    module_files = heatmap_data[heatmap_data['Module'] == module].head(8)
    viz_files.append(module_files)

viz_data = pd.concat(viz_files)

# Create matrix for heatmap (files x metrics)
heatmap_matrix = viz_data[['Name', 'SumCyclomatic', 'CountLine', 'RatioCommentToCode']].copy()
heatmap_matrix['RatioCommentToCode'] = heatmap_matrix['RatioCommentToCode'] * 100  # Convert to percentage
heatmap_matrix = heatmap_matrix.set_index('Name')

# Normalize for better visualization
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
heatmap_normalized = pd.DataFrame(
    scaler.fit_transform(heatmap_matrix),
    columns=['Cyclomatic Complexity', 'Lines of Code', 'Comment %'],
    index=heatmap_matrix.index
)

# Create heatmap
fig = px.imshow(
    heatmap_normalized.T,
    labels=dict(x="File", y="Metric", color="Normalized Value"),
    x=heatmap_normalized.index,
    y=heatmap_normalized.columns,
    color_continuous_scale='RdYlGn_r',  # Red = high, Green = low
    title='RQ5: Complexity Heatmap<br><sub>Red = High complexity/size | Green = Low</sub>',
    aspect='auto'
)

fig.update_xaxes(tickangle=-45, tickfont=dict(size=8))
fig.update_layout(height=400)
fig.show()

print("✓ Heatmap reveals complexity hotspots")
print(f"  Files in red require attention")
print(f"  Files in green are well-optimized")

### RQ5 Visualization 2: Function Complexity Analysis

Detailed breakdown of function-level complexity patterns.

In [ ]:
# Comprehensive function complexity analysis
print("Analyzing function-level complexity patterns...")

# Create multi-dimensional function analysis
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Complexity vs Nesting Depth',
        'Complexity vs Function Size',
        'Complexity vs Parameter Count',
        'Top 15 Most Complex Functions'
    ),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'bar'}]]
)

# 1. Complexity vs Nesting
fig.add_trace(go.Scatter(
    x=function_complexity['MaxNesting'],
    y=function_complexity['Cyclomatic'],
    mode='markers',
    marker=dict(size=5, color=function_complexity['CountLine'],
                colorscale='Viridis', showscale=False, opacity=0.6),
    text=function_complexity['Name'],
    hovertemplate='%{text}<br>Nesting: %{x}<br>Complexity: %{y}<extra></extra>'
), row=1, col=1)

# 2. Complexity vs Size
fig.add_trace(go.Scatter(
    x=function_complexity['CountLine'],
    y=function_complexity['Cyclomatic'],
    mode='markers',
    marker=dict(size=5, color=function_complexity['MaxNesting'],
                colorscale='Reds', showscale=False, opacity=0.6),
    text=function_complexity['Name'],
    hovertemplate='%{text}<br>LOC: %{x}<br>Complexity: %{y}<extra></extra>'
), row=1, col=2)

# 3. Complexity vs Parameters
fig.add_trace(go.Scatter(
    x=function_complexity['CountInput'],
    y=function_complexity['Cyclomatic'],
    mode='markers',
    marker=dict(size=5, color='lightblue', opacity=0.6),
    text=function_complexity['Name'],
    hovertemplate='%{text}<br>Parameters: %{x}<br>Complexity: %{y}<extra></extra>'
), row=2, col=1)

# 4. Top complex functions
top_complex = function_complexity.nlargest(15, 'Cyclomatic')
fig.add_trace(go.Bar(
    x=top_complex['Cyclomatic'],
    y=top_complex['Name'],
    orientation='h',
    marker_color='indianred'
), row=2, col=2)

# Update axes
fig.update_xaxes(title_text="Max Nesting Depth", row=1, col=1)
fig.update_xaxes(title_text="Lines of Code", row=1, col=2)
fig.update_xaxes(title_text="Parameter Count", row=2, col=1)
fig.update_xaxes(title_text="Cyclomatic Complexity", row=2, col=2)

fig.update_yaxes(title_text="Cyclomatic Complexity", row=1, col=1)
fig.update_yaxes(title_text="Cyclomatic Complexity", row=1, col=2)
fig.update_yaxes(title_text="Cyclomatic Complexity", row=2, col=1)
fig.update_yaxes(tickfont=dict(size=8), row=2, col=2)

fig.update_layout(height=800, showlegend=False,
                  title_text='RQ5: Comprehensive Function Complexity Analysis')
fig.show()

# Statistical summary
print("\nFunction Complexity Correlations:")
print(f"  Complexity vs Nesting:    r = {function_complexity['Cyclomatic'].corr(function_complexity['MaxNesting']):.3f}")
print(f"  Complexity vs Size:       r = {function_complexity['Cyclomatic'].corr(function_complexity['CountLine']):.3f}")
print(f"  Complexity vs Parameters: r = {function_complexity['Cyclomatic'].corr(function_complexity['CountInput']):.3f}")

print(f"\n✓ Nesting depth {'strongly' if function_complexity['Cyclomatic'].corr(function_complexity['MaxNesting']) > 0.5 else 'moderately'} correlates with complexity")

### RQ5 Visualization 3: File Size Distribution

Analysis of how file size relates to complexity and maintainability.

In [ ]:
# File size distribution analysis
print("Analyzing file size distribution patterns...")

# Create comprehensive size analysis
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'File Size Distribution',
        'Size vs Complexity',
        'Size vs Comment Ratio',
        'Size Categories'
    ),
    specs=[[{'type': 'histogram'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'pie'}]]
)

# 1. Histogram of file sizes
fig.add_trace(go.Histogram(
    x=file_complexity['CountLine'],
    nbinsx=30,
    marker_color='lightblue',
    name='Files'
), row=1, col=1)

# 2. Size vs Complexity scatter
fig.add_trace(go.Scatter(
    x=file_complexity['CountLine'],
    y=file_complexity['SumCyclomatic'],
    mode='markers',
    marker=dict(
        size=file_complexity['CountDeclFunction']*2,
        color=file_complexity['RatioCommentToCode'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Comment<br>Ratio', x=0.46, len=0.4)
    ),
    text=file_complexity['Name'],
    hovertemplate='%{text}<br>LOC: %{x}<br>Complexity: %{y}<extra></extra>'
), row=1, col=2)

# 3. Size vs Comment ratio
fig.add_trace(go.Scatter(
    x=file_complexity['CountLine'],
    y=file_complexity['RatioCommentToCode']*100,
    mode='markers',
    marker=dict(size=8, color='coral', opacity=0.6),
    text=file_complexity['Name'],
    hovertemplate='%{text}<br>LOC: %{x}<br>Comments: %{y:.1f}%<extra></extra>'
), row=2, col=1)

# 4. Size categories pie chart
size_categories = pd.cut(
    file_complexity['CountLine'],
    bins=[0, 200, 500, 1000, 10000],
    labels=['Small (<200)', 'Medium (200-500)', 'Large (500-1000)', 'Very Large (>1000)']
)
size_dist = size_categories.value_counts()

fig.add_trace(go.Pie(
    labels=size_dist.index,
    values=size_dist.values,
    marker=dict(colors=['lightgreen', 'lightyellow', 'lightcoral', 'darkred'])
), row=2, col=2)

# Update axes
fig.update_xaxes(title_text="Lines of Code", row=1, col=1)
fig.update_xaxes(title_text="Lines of Code", row=1, col=2)
fig.update_xaxes(title_text="Lines of Code", row=2, col=1)

fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Cyclomatic Complexity", row=1, col=2)
fig.update_yaxes(title_text="Comment Ratio (%)", row=2, col=1)

fig.update_layout(height=800, showlegend=False,
                  title_text='RQ5: File Size Distribution Analysis')
fig.show()

print(f"\nFile Size Statistics:")
print(f"  Mean size: {file_complexity['CountLine'].mean():.0f} LOC")
print(f"  Median size: {file_complexity['CountLine'].median():.0f} LOC")
print(f"  Largest file: {file_complexity['CountLine'].max():.0f} LOC")
print(f"\nSize Distribution:")
for cat, count in size_dist.items():
    print(f"  {cat:25s}: {count:2d} files ({count/len(file_complexity)*100:.1f}%)")

print(f"\n✓ Most files are compact (<500 LOC), ideal for memory-constrained development")

### RQ5 Visualization 4: Statistical Distribution Analysis

Box plots showing distribution of key complexity metrics.

In [ ]:
# Statistical distribution analysis with box plots
print("Creating statistical distribution visualizations...")

# Prepare data for box plots
box_data = file_complexity[['Category', 'SumCyclomatic', 'CountLine',
                             'RatioCommentToCode', 'CountDeclFunction']].copy()

# Create box plots for each metric by category
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Complexity by Category',
        'File Size by Category',
        'Comment Ratio by Category',
        'Function Count by Category'
    )
)

categories = box_data['Category'].value_counts().head(6).index

# 1. Complexity distribution
for cat in categories:
    cat_data = box_data[box_data['Category'] == cat]
    fig.add_trace(go.Box(
        y=cat_data['SumCyclomatic'],
        name=cat,
        showlegend=False
    ), row=1, col=1)

# 2. Size distribution
for cat in categories:
    cat_data = box_data[box_data['Category'] == cat]
    fig.add_trace(go.Box(
        y=cat_data['CountLine'],
        name=cat,
        showlegend=False
    ), row=1, col=2)

# 3. Comment ratio distribution
for cat in categories:
    cat_data = box_data[box_data['Category'] == cat]
    fig.add_trace(go.Box(
        y=cat_data['RatioCommentToCode']*100,
        name=cat,
        showlegend=False
    ), row=2, col=1)

# 4. Function count distribution
for cat in categories:
    cat_data = box_data[box_data['Category'] == cat]
    fig.add_trace(go.Box(
        y=cat_data['CountDeclFunction'],
        name=cat,
        showlegend=False
    ), row=2, col=2)

fig.update_yaxes(title_text="Cyclomatic Complexity", row=1, col=1)
fig.update_yaxes(title_text="Lines of Code", row=1, col=2)
fig.update_yaxes(title_text="Comment Ratio (%)", row=2, col=1)
fig.update_yaxes(title_text="Function Count", row=2, col=2)

fig.update_xaxes(tickangle=-45, row=1, col=1)
fig.update_xaxes(tickangle=-45, row=1, col=2)
fig.update_xaxes(tickangle=-45, row=2, col=1)
fig.update_xaxes(tickangle=-45, row=2, col=2)

fig.update_layout(height=800,
                  title_text='RQ5: Statistical Distribution Analysis by Functional Category')
fig.show()

print("✓ Box plots reveal:")
print("  - Median values (line in box)")
print("  - Interquartile range (box)")
print("  - Outliers (individual points)")
print("\nCategories with wide boxes show HIGH VARIABILITY in complexity")

### RQ5 Visualization 5: Comparative Performance Tables

Top and bottom performers across different metrics.

In [ ]:
# Create comparative analysis tables
print("Creating comparative performance tables...\n")

# Most complex files
print("="*80)
print("TOP 10 MOST COMPLEX FILES")
print("="*80)
most_complex = file_complexity.nlargest(10, 'SumCyclomatic')[
    ['Name', 'SumCyclomatic', 'CountLine', 'CountDeclFunction', 'Category']
]
print(most_complex.to_string(index=False))

# Largest files
print("\n" + "="*80)
print("TOP 10 LARGEST FILES")
print("="*80)
largest_files = file_complexity.nlargest(10, 'CountLine')[
    ['Name', 'CountLine', 'SumCyclomatic', 'RatioCommentToCode', 'Category']
]
print(largest_files.to_string(index=False))

# Best documented
print("\n" + "="*80)
print("TOP 10 BEST DOCUMENTED FILES")
print("="*80)
file_complexity['CommentDensity'] = file_complexity['CountLineComment'] / file_complexity['CountLine']
best_documented = file_complexity.nlargest(10, 'CommentDensity')[
    ['Name', 'CommentDensity', 'CountLine', 'SumCyclomatic', 'Category']
].copy()
best_documented['CommentDensity'] = best_documented['CommentDensity'].round(2)
print(best_documented.to_string(index=False))

# Most functions
print("\n" + "="*80)
print("TOP 10 FILES WITH MOST FUNCTIONS")
print("="*80)
most_functions = file_complexity.nlargest(10, 'CountDeclFunction')[
    ['Name', 'CountDeclFunction', 'CountLine', 'SumCyclomatic', 'Category']
]
print(most_functions.to_string(index=False))

# Create visual comparison
comparison_data = pd.DataFrame({
    'File': most_complex.head(5)['Name'],
    'Complexity Rank': range(1, 6),
    'Complexity': most_complex.head(5)['SumCyclomatic'].values,
    'Size Rank': [list(largest_files['Name']).index(f)+1 if f in list(largest_files['Name']) else None
                  for f in most_complex.head(5)['Name']],
    'Doc Rank': [list(best_documented['Name']).index(f)+1 if f in list(best_documented['Name']) else None
                 for f in most_complex.head(5)['Name']]
})

# Visualize ranking comparison
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[1]*5, y=comparison_data['Complexity Rank'],
    mode='markers+text',
    marker=dict(size=15, color='red'),
    text=comparison_data['File'].str.split('.').str[0],
    textposition='middle right',
    name='Complexity Rank',
    showlegend=False
))

fig.add_trace(go.Scatter(
    x=[2]*5, y=comparison_data['Size Rank'].fillna(15),
    mode='markers',
    marker=dict(size=15, color='blue'),
    name='Size Rank',
    showlegend=False
))

fig.add_trace(go.Scatter(
    x=[3]*5, y=comparison_data['Doc Rank'].fillna(15),
    mode='markers',
    marker=dict(size=15, color='green'),
    name='Doc Rank',
    showlegend=False
))

# Add connecting lines
for idx, row in comparison_data.iterrows():
    fig.add_trace(go.Scatter(
        x=[1, 2, 3],
        y=[row['Complexity Rank'], row['Size Rank'] if pd.notna(row['Size Rank']) else 15,
           row['Doc Rank'] if pd.notna(row['Doc Rank']) else 15],
        mode='lines',
        line=dict(color='lightgray', width=1),
        showlegend=False,
        hoverinfo='skip'
    ))

fig.update_layout(
    title='RQ5: Ranking Comparison - Top 5 Most Complex Files<br><sub>How do they rank in size and documentation?</sub>',
    xaxis=dict(
        tickmode='array',
        tickvals=[1, 2, 3],
        ticktext=['Complexity', 'Size', 'Documentation'],
        range=[0.5, 3.5]
    ),
    yaxis=dict(
        title='Rank (1=best/highest)',
        autorange='reversed',
        range=[0, 16]
    ),
    height=500
)

fig.show()

print("\n✓ Tables show performance across multiple dimensions")
print("✓ Most complex files are not necessarily the largest")
print("✓ Documentation varies independently of complexity")